# Monthly forecast accuracy pack (legacy)

Trading Analytics. Used for the packs from July 2025 to June 2026.

Run on KPI close day, once Billing has delivered the month's charge-basis snapshot, then record the close in
`kpi_close_log` and circulate the pack for sign-off.

Inputs (old analytics warehouse):
- `billing.charge_basis_snapshot` (snapshot_month, region, portfolio, delivery_date, mwh), requested from Billing for
  the month;
- `trading.locked_forecasts` (model, run_date, region, portfolio, target_date, mwh), from the trading system.

In [ ]:
import pandas as pd
import sqlalchemy as sa

MONTH = "2026-06"
engine = sa.create_engine("postgresql+psycopg://analytics-ro@awh-prod.internal/analytics")

In [ ]:
examples = pd.read_sql_query(sa.text("""
    SELECT f.model, f.run_date, f.region, f.portfolio, f.target_date,
           (f.target_date - f.run_date) AS horizon,
           f.mwh AS forecast_mwh,
           s.mwh AS actual_mwh
    FROM trading.locked_forecasts f
    LEFT JOIN billing.charge_basis_snapshot s
           ON s.snapshot_month = :month
          AND s.region = f.region
          AND s.portfolio = f.portfolio
          AND s.delivery_date = f.target_date
    WHERE to_char(f.target_date, 'YYYY-MM') = :month
"""), engine, params={"month": MONTH})
examples["scored"] = examples["actual_mwh"].notna()
examples["abs_error_mwh"] = (examples["forecast_mwh"] - examples["actual_mwh"]).abs()

In [ ]:
def wape(df):
    s = df[df["scored"]]
    return s["abs_error_mwh"].sum() / s["actual_mwh"].sum()

pack = (examples.groupby(["model", "portfolio"])
        .apply(lambda g: pd.Series({"wape_pct": round(100 * wape(g), 1)}))
        .reset_index())
by_horizon = examples.groupby(["model", "portfolio", "horizon"]).apply(wape).rename("wape").reset_index()
pack

In [ ]:
# head-to-head on common forecasts
unit = ["region", "portfolio", "target_date", "horizon"]
scored = examples[examples["scored"]]
models = sorted(scored["model"].unique())
rows = []
for i, a in enumerate(models):
    for b in models[i + 1:]:
        both = scored[scored["model"] == a].merge(scored[scored["model"] == b], on=unit, suffixes=("_a", "_b"))
        if len(both):
            wa = both["abs_error_mwh_a"].sum() / both["actual_mwh_a"].sum()
            wb = both["abs_error_mwh_b"].sum() / both["actual_mwh_b"].sum()
            rows.append((a, b, len(both), wa, wb, wb / wa - 1))
pd.DataFrame(rows, columns=["model_a", "model_b", "forecasts", "wape_a", "wape_b", "b_vs_a"])

In [ ]:
pack.to_csv(f"packs/{MONTH}_pack.csv", index=False)
by_horizon.to_csv(f"packs/{MONTH}_by_horizon.csv", index=False)